In [2]:
# ============================================================
# CELL 0 – LOCAL ENVIRONMENT SETUP
# ============================================================
import os, sys

# Cấu trúc Local-first: lấy thư mục gốc của project
# Vì notebook nằm trong thư mục notebooks/, nên PROJECT_ROOT là thư mục cha.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
REPO_PATH = PROJECT_ROOT

# Add src/ to Python path
if os.path.join(PROJECT_ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print('✅ Local Environment ready. PROJECT_ROOT:', PROJECT_ROOT)


✅ Local Environment ready. PROJECT_ROOT: d:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection


In [3]:
# ============================================================
# CELL 1 - PREPROCESSING LOGIC
# ============================================================
import os
import json
from omegaconf import OmegaConf

from data.preprocessing import (
    compute_balanced_class_weights,
    process_split,
    resolve_project_path,
    save_class_weights,
)

cfg = OmegaConf.load(os.path.join(REPO_PATH, "configs/paths.yaml"))
project_root = PROJECT_ROOT
num_labels = int(cfg.model.num_labels)

split_config = {
    "train": (cfg.data.train_raw, cfg.data.train_processed),
    "dev": (cfg.data.dev_raw, cfg.data.val_processed),
    "test": (cfg.data.test_raw, cfg.data.test_processed),
}

processed = {}
for split_name, (raw_template, processed_template) in split_config.items():
    raw_path = resolve_project_path(raw_template, project_root)
    out_path = resolve_project_path(processed_template, project_root)
    if not raw_path.exists():
        print(f"Warning: File not found {raw_path}")
        continue
    processed[split_name] = process_split(raw_path, out_path)
    print(f"Processed {split_name}: {out_path} ({len(processed[split_name]):,} rows)")

if "train" in processed:
    class_weights = compute_balanced_class_weights(processed["train"]["label"], num_labels)
    class_weights_path = resolve_project_path(cfg.results.class_weights, project_root)
    save_class_weights(class_weights, class_weights_path)
    print("Class weights from train split only:", class_weights)
    print(f"Class weights saved: {class_weights_path}")
else:
    print("Cannot compute class weights: train split was not processed.")


Processed train: D:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection\data\processed\train.parquet (23,973 rows)
Processed dev: D:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection\data\processed\dev.parquet (2,662 rows)
Processed test: D:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection\data\processed\test.parquet (6,650 rows)
Class weights from train split only: {0: 0.4033414092469211, 1: 4.978816199376947, 2: 3.1263693270735526}
Class weights saved: D:\bt_lap_trinh\NLP\Ứng dụng\Project\hate-speech-detection\results\class_weights.json
